# Four-state incomplete-competition model in true free receptor

This notebook derives and validates the quintic used by `bindcurve` for the four-state competitive-binding models.

`bindcurve` solves the quintic directly in free receptor concentration

$$
R \equiv [R].
$$

The response is then computed from the actual four-state bound tracer population, `RS + RLS`.

Reference:

Roehrl, M. H. A.; Wang, J. Y.; Wagner, G. *Biochemistry* **2004**, 43(51), 16056-16066. DOI: [10.1021/bi048233g](https://doi.org/10.1021/bi048233g)


## Species model

The four-state incomplete-competition model uses free species

$$
R,\quad S \equiv L^*,\quad L
$$

and complexes

$$
RS,\quad RL,\quad RLS.
$$

The equilibrium concentration expressions are

$$
[RS] = \frac{[R][S]}{K_d^*},
$$

$$
[RL] = \frac{[R][L]}{K_d},
$$

$$
[RLS] = \frac{[R][L][S]}{K_dK_{d3}}.
$$

Here `Kds` in the source code means $K_d^*$, the tracer dissociation constant.

The mass balances are

$$
R_T = R + RS + RL + RLS,
$$

$$
S_T = S + RS + RLS,
$$

$$
L_T = L + RL + RLS.
$$

The experimental tracer-bound fraction is

$$
F_b^* = \frac{RS + RLS}{S_T}.
$$


## Symbolic derivation of the quintic in free receptor

The following cell eliminates free tracer `S` and free competitor `L` from the three mass-balance equations, leaving one polynomial in true free receptor `R`.


In [ ]:
import sympy as sp

R, S, L = sp.symbols("R S L")
RT, ST, LT = sp.symbols("RT ST LT")
Kds, Kd, Kd3 = sp.symbols("Kds Kd Kd3")

RS = R * S / Kds
RL = R * L / Kd
RLS = R * L * S / (Kd * Kd3)

eq_receptor = RT - R - RS - RL - RLS
eq_tracer = ST - S - RS - RLS
eq_competitor = LT - L - RL - RLS

common_denominator = Kds * Kd * Kd3
p_receptor = sp.expand(eq_receptor * common_denominator)
p_tracer = sp.expand(eq_tracer * common_denominator)
p_competitor = sp.expand(eq_competitor * common_denominator)

relation_ligand = sp.resultant(p_tracer, p_competitor, S)
relation_receptor = sp.resultant(p_receptor, p_tracer, S)

relation_ligand = sp.factor(relation_ligand / (Kd3 * Kds))
relation_receptor = sp.factor(-relation_receptor / (Kd3 * Kds))

resultant = sp.resultant(relation_ligand, relation_receptor, L)
factored = sp.factor(resultant)

factored


The resultant contains extraneous factors from the elimination. The physical receptor polynomial is the quintic factor.


In [ ]:
extraneous = -Kd3 * Kd**4 * Kds**2 * R**2 * ST
quintic = sp.factor(factored / extraneous)
poly = sp.Poly(quintic, R)

assert poly.degree() == 5

coefficients = poly.all_coeffs()
coefficients


Written as

$$
aR^5 + bR^4 + cR^3 + dR^2 + eR + f = 0,
$$

the coefficients are the expressions below.


In [ ]:
for symbol, coefficient in zip("abcdef", coefficients, strict=True):
    print(f"{symbol} =")
    print(sp.factor(coefficient))
    print()


## Runtime coefficient helper

The implementation in `bindcurve.modeling.competitive._competitive_four_state_coefficients` should match the symbolic coefficients above. This cell verifies that correspondence symbolically.


In [ ]:
runtime_coefficients = [
    Kds - Kd3,
    (
        -Kd3 * Kd
        - Kd3 * Kds
        - Kd3 * LT
        + Kd3 * RT
        - Kd3 * ST
        + Kd * Kds
        + Kds**2
        + Kds * LT
        - 2 * Kds * RT
        + Kds * ST
    ),
    (
        Kd3 * Kd * RT
        - Kd3 * Kd * ST
        - Kd3 * Kds * LT
        + Kd3 * Kds * RT
        + Kd * Kds**2
        - 2 * Kd * Kds * RT
        + 2 * Kd * Kds * ST
        + 2 * Kds**2 * LT
        - 2 * Kds**2 * RT
        - Kds * LT * RT
        + Kds * LT * ST
        + Kds * RT**2
        - Kds * RT * ST
    ),
    (
        Kd3 * Kd**2 * Kds
        + Kd3 * Kd * Kds**2
        + Kd3 * Kd * Kds * LT
        + Kd3 * Kd * Kds * ST
        + Kd * Kds**2 * LT
        - 2 * Kd * Kds**2 * RT
        + Kd * Kds**2 * ST
        + Kd * Kds * RT**2
        - 2 * Kd * Kds * RT * ST
        + Kd * Kds * ST**2
        + Kds**2 * LT**2
        - 2 * Kds**2 * LT * RT
        + Kds**2 * RT**2
    ),
    (
        Kd3 * Kd**2 * Kds**2
        - Kd3 * Kd**2 * Kds * RT
        + Kd3 * Kd**2 * Kds * ST
        + Kd3 * Kd * Kds**2 * LT
        - Kd3 * Kd * Kds**2 * RT
        - Kd * Kds**2 * LT * RT
        + Kd * Kds**2 * LT * ST
        + Kd * Kds**2 * RT**2
        - Kd * Kds**2 * RT * ST
    ),
    -Kd3 * Kd**2 * Kds**2 * RT,
]

assert all(
    sp.simplify(symbolic - runtime) == 0
    for symbolic, runtime in zip(coefficients, runtime_coefficients, strict=True)
)


## Numeric root selection

For each competitor concentration, `bindcurve` evaluates the coefficient helper, finds all numeric roots, and keeps the effectively real root in the physical free-receptor interval

$$
0 \le R \le R_T.
$$

If `Kds == Kd3`, the leading coefficient is zero and the quintic degenerates to a quartic. The runtime helper trims leading near-zero coefficients before calling `numpy.roots`.


In [ ]:
import numpy as np

from bindcurve.modeling.competitive import (
    _competitive_four_state_coefficients,
    _select_physical_root,
)

params = {
    "RT": 0.05,
    "LsT": 0.005,
    "Kds": 0.02,
    "Kd": 1.6,
    "Kd3": 0.5,
}

ligand_total = 0.1
numeric_coefficients = _competitive_four_state_coefficients(
    ligand_total,
    **params,
)

R_free = _select_physical_root(
    numeric_coefficients,
    lower_bound=0.0,
    upper_bound=params["RT"],
)

print("coefficients:", numeric_coefficients)
print("roots:", np.roots(numeric_coefficients))
print("selected R:", R_free)
print("residual:", np.polyval(numeric_coefficients, R_free))
assert 0.0 <= R_free <= params["RT"]
assert np.isclose(np.polyval(numeric_coefficients, R_free), 0.0)


## Bound tracer fraction from true free receptor

For a fixed free receptor concentration `R`, the remaining free concentrations can be recovered without another nonlinear solve.

Define

$$
A = 1 + \frac{R}{K_d^*},
$$

$$
B = 1 + \frac{R}{K_d},
$$

$$
C = \frac{R}{K_dK_{d3}}.
$$

The tracer balance gives

$$
S_T = S(A + CL),
$$

and the competitor balance gives

$$
L_T = L(B + CS).
$$

Substituting the first into the second gives a quadratic in free competitor `L`:

$$
BC L^2 + (AB + CS_T - CL_T)L - AL_T = 0.
$$

After selecting the non-negative root, the tracer-bound fraction can be evaluated as

$$
F_b^* = \frac{RS + RLS}{S_T}
      = \frac{R/K_d^* + CL}{1 + R/K_d^* + CL}.
$$

In source-code names, this is implemented as `R/Kds + C*L`, where `C = R/(Kd*Kd3)`.


In [ ]:
def species_from_free_receptor(R_free, ligand_total, *, RT, LsT, Kds, Kd, Kd3):
    a = 1.0 + R_free / Kds
    b = 1.0 + R_free / Kd
    c = R_free / (Kd * Kd3)

    quadratic_a = b * c
    quadratic_b = a * b + c * LsT - c * ligand_total
    quadratic_c = -a * ligand_total

    discriminant = quadratic_b**2 - 4.0 * quadratic_a * quadratic_c
    if abs(quadratic_a) > np.finfo(float).tiny:
        L_free = (-quadratic_b + np.sqrt(discriminant)) / (2.0 * quadratic_a)
    else:
        L_free = ligand_total / b

    S_free = LsT / (a + c * L_free)

    RS = R_free * S_free / Kds
    RL = R_free * L_free / Kd
    RLS = R_free * L_free * S_free / (Kd * Kd3)

    return {
        "R": R_free,
        "S": S_free,
        "L": L_free,
        "RS": RS,
        "RL": RL,
        "RLS": RLS,
    }


species = species_from_free_receptor(R_free, ligand_total, **params)
fraction_bound = (species["RS"] + species["RLS"]) / params["LsT"]

print(species)
print("FSB:", fraction_bound)

assert np.isclose(
    species["R"] + species["RS"] + species["RL"] + species["RLS"],
    params["RT"],
)
assert np.isclose(
    species["S"] + species["RS"] + species["RLS"],
    params["LsT"],
)
assert np.isclose(
    species["L"] + species["RL"] + species["RLS"],
    ligand_total,
)


## End-to-end model evaluation

The model response is

$$
y = ymin + (ymax - ymin)F_b^*.
$$

The total/nonspecific variant does not need a separate algebraic derivation. It uses the same free-receptor quintic, but with

$$
K_{d,\mathrm{eff}} = (1 + N)K_d.
$$


In [ ]:
from bindcurve.modeling.competitive import (
    CompetitiveFourStateSpecificKdModel,
    CompetitiveFourStateTotalKdModel,
)

concentration = np.logspace(-3, 2, 8)
specific_model = CompetitiveFourStateSpecificKdModel()
total_model = CompetitiveFourStateTotalKdModel()

specific = specific_model.evaluate(
    concentration,
    ymin=5.0,
    ymax=95.0,
    **params,
)
total_n0 = total_model.evaluate(
    concentration,
    ymin=5.0,
    ymax=95.0,
    N=0.0,
    **params,
)

N = 0.35
total = total_model.evaluate(
    concentration,
    ymin=5.0,
    ymax=95.0,
    N=N,
    **params,
)
specific_with_effective_kd = specific_model.evaluate(
    concentration,
    ymin=5.0,
    ymax=95.0,
    RT=params["RT"],
    LsT=params["LsT"],
    Kds=params["Kds"],
    Kd=(1.0 + N) * params["Kd"],
    Kd3=params["Kd3"],
)

assert np.allclose(total_n0, specific)
assert np.allclose(total, specific_with_effective_kd)

np.column_stack([concentration, specific, total])


## Takeaway

The clean four-state implementation is now:

1. Derive and solve the quintic in true free receptor concentration `R`.
2. Select the physical root in `0 <= R <= RT`.
3. Compute the observable tracer-bound fraction from the actual species `RS + RLS`.
4. Use the same machinery for `comp_4st_total`, with `Kd` replaced by `(1 + N) * Kd`.

This keeps the four-state implementation consistent with the three-state implementation, where the primary equilibrium coordinate is also true free receptor concentration.
